In [ ]:

# @title Step 1 — Clean, clone repository, and list files

%cd /content

import os
import shutil
import subprocess

project_dir = "/content/b2-multi-account-worker"

# 1. Delete previous project folder safely
if os.path.exists(project_dir):
    shutil.rmtree(project_dir)
    print("Previous project folder deleted.")
else:
    print("No previous project folder found.")

# 2. Ask for the GitHub repository URL
git_url = input("Enter your GitHub repository URL: ").strip()

if not git_url:
    raise ValueError("GitHub URL cannot be empty.")

# Extract plain URL if a Markdown link was pasted
import re
match = re.search(r"https?://github\.com/[^)\]\s]+", git_url)
if match:
    git_url = match.group(0)

print(f"\nCloning: {git_url}")

# 3. Clone into the project folder
result = subprocess.run([
    "git", "clone", git_url, "b2-multi-account-worker"
])

if result.returncode != 0:
    raise RuntimeError("Cloning failed. Check the Git error above.")

# 4. Change directory and list files
%cd /content/b2-multi-account-worker
!ls -la


In [ ]:
# @title Check the worker name
%cd /content/b2-multi-account-worker
!cat wrangler.toml

In [ ]:

# @title Change the Worker name
%cd /content/b2-multi-account-worker

from pathlib import Path
import re

p = Path("wrangler.toml")

if not p.is_file():
    raise FileNotFoundError("wrangler.toml not found.")

config = p.read_text(encoding="utf-8")

# Ask for the desired Worker name
worker_name = input("Enter the desired Worker name: ").strip()

# Validate the name
if not worker_name:
    raise ValueError("Worker name cannot be empty.")

if not re.fullmatch(r"[a-zA-Z0-9_-]+", worker_name):
    raise ValueError(
        "Invalid name. Use only letters, numbers, hyphens, and underscores."
    )

# Update the name field
updated_config, count = re.subn(
    r'(?m)^name\s*=\s*".*?"\s*$',
    f'name = "{worker_name}"',
    config,
    count=1
)

if count == 0:
    raise ValueError('Could not find the Worker name in wrangler.toml.')

# Save the updated configuration
p.write_text(updated_config, encoding="utf-8")

print("\nWorker name updated successfully.")
print(f"New Worker name: {worker_name}")
print("\n========== Updated wrangler.toml ==========\n")
print(p.read_text(encoding="utf-8"))


In [ ]:
# @title Cloudflare login (fill api value)
import os
import getpass

os.environ["CLOUDFLARE_API_TOKEN"] = getpass.getpass(
    "Enter your Cloudflare API Token: "
)

print("API token loaded for this Colab session.")

In [ ]:
# @title Wrangler install
!npm install -g wrangler
!wrangler whoami

In [ ]:
# @title Upload → rename → validate → upload to Cloudflare a .json or .txt file containing B2 ACCOUNT DETAILS
from google.colab import files
from pathlib import Path
import json
import subprocess

# 1. Confirm the Worker project directory
project_dir = Path("/content/b2-multi-account-worker")
wrangler_file = project_dir / "wrangler.toml"

if not wrangler_file.is_file():
    raise FileNotFoundError(
        f"wrangler.toml not found at {project_dir}. "
        "Check your project directory."
    )

import os
os.chdir(project_dir)

# 2. Upload one JSON or TXT file
uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one .json or .txt file.")

original_name = next(iter(uploaded))

if not original_name.lower().endswith((".json", ".txt")):
    raise ValueError("Only .json and .txt files are supported.")

# 3. Rename to the standard filename
source = Path(original_name)
target = project_dir / "b2_accounts.json"

if source.resolve() != target.resolve():
    source.replace(target)

print(f"File saved as: {target.name}")

# 4. Validate JSON and required account fields
config_text = target.read_text(encoding="utf-8")
accounts = json.loads(config_text)

if not isinstance(accounts, dict) or not accounts:
    raise ValueError("The file must contain a non-empty JSON object.")

required = {"endpoint", "region", "keyId", "applicationKey"}

for name, account in accounts.items():
    if not isinstance(account, dict):
        raise ValueError(f"{name} must be a JSON object.")

    missing = required - account.keys()
    if missing:
        raise ValueError(f"{name} is missing: {sorted(missing)}")

    for field in required:
        value = account[field]
        if not isinstance(value, str) or not value.strip():
            raise ValueError(
                f"{name}: {field} must be a non-empty string."
            )

print(f"Validation successful: {len(accounts)} account(s).")

# 5. Upload the complete JSON as the Worker secret
result = subprocess.run(
    ["wrangler", "secret", "put", "B2_ACCOUNTS"],
    input=config_text,
    text=True,
    capture_output=True
)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("Cloudflare secret upload failed.")

print(result.stdout)
print("B2_ACCOUNTS secret updated successfully.")

# 6. Clear references to sensitive configuration
accounts = None
config_text = None
uploaded = None


In [ ]:
# @title Deploy worker to cloudflare
%cd /content/b2-multi-account-worker
!wrangler deploy

In [ ]:

# @title Optional — Change CACHE_TTL and deploy
from pathlib import Path
import re
import subprocess

project_dir = Path("/content/b2-multi-account-worker")
config_file = project_dir / "wrangler.toml"

if not config_file.is_file():
    raise FileNotFoundError("wrangler.toml not found. Clone your repository first.")

answer = input("Do you want to change CACHE_TTL? (y/n): ").strip().lower()

if answer in ("y", "yes"):
    ttl_input = input(
        "Enter CACHE_TTL in seconds (e.g. 3600, 21600, 86400): "
    ).strip()

    if not ttl_input.isdigit():
        raise ValueError("CACHE_TTL must be a whole number of seconds.")

    new_ttl = int(ttl_input)

    config = config_file.read_text(encoding="utf-8")

    # Find the [vars] section
    match = re.search(r"(?m)^\[vars\]\s*$", config)
    if not match:
        raise ValueError("No [vars] section found in wrangler.toml.")

    # Update an existing active or commented CACHE_TTL entry in [vars],
    # or insert a new entry if none exists.
    start = match.end()
    next_section = re.search(r"(?m)^\[", config[start:])
    end = start + next_section.start() if next_section else len(config)
    vars_section = config[start:end]

    ttl_pattern = r'(?m)^\s*#?\s*CACHE_TTL\s*=.*$'

    if re.search(ttl_pattern, vars_section):
        vars_section = re.sub(
            ttl_pattern,
            f'CACHE_TTL = "{new_ttl}"',
            vars_section,
            count=1
        )
    else:
        vars_section = f'\nCACHE_TTL = "{new_ttl}"' + vars_section

    config = config[:start] + vars_section + config[end:]
    config_file.write_text(config, encoding="utf-8")

    print("\nUpdated wrangler.toml:\n")
    print(config_file.read_text(encoding="utf-8"))

    deploy = input("Deploy this change to Cloudflare now? (y/n): ").strip().lower()

    if deploy in ("y", "yes"):
        result = subprocess.run(
            ["wrangler", "deploy"],
            cwd=str(project_dir),
            text=True
        )
        if result.returncode != 0:
            raise RuntimeError("Deployment failed. Check the output above.")

        print(f"\nDeployment completed with CACHE_TTL = {new_ttl} seconds.")
    else:
        print("Configuration saved locally. No deployment performed.")

elif answer in ("n", "no"):
    print("No changes made to CACHE_TTL.")
else:
    raise ValueError("Please enter y or n.")
